# Notebook para el ejercicio 2 del primer boletín

In [2]:
using DelimitedFiles
using Statistics
using Flux
using Flux.Losses


## *Parte 1*: Implementar One Hot Enconding

In [266]:
function oneHotEncoding(feature:: AbstractArray{<:Any, 1}, classes:: AbstractArray{<:Any, 1})

    num_classes = length(classes)

    if num_classes <= 2
        oneHot = Matrix{Bool}(undef, length(feature), 1)
        oneHot[:, 1] = feature .== classes[1]
    else
        oneHot = Matrix{Bool}(undef, length(feature), num_classes)
        for i in 1:num_classes
            oneHot[:, i] = feature .== classes[i]
        end
    end

    return oneHot
end


oneHotEncoding (generic function with 2 methods)

In [269]:
feature = [0, 1, 2, 2, 0, 1, 5, 2, 3, 1, 5, 5, 4, 3, 4, 5]
feature_2_cls = [3, 3, 2, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3]
classes = unique(feature)

oneHot = oneHotEncoding(feature_2_cls, unique(feature_2_cls))

13×1 Matrix{Bool}:
 1
 1
 0
 0
 1
 0
 1
 0
 1
 0
 1
 0
 1

### Overload de oneHotEncoding para diferentes situaciones

In [6]:
function oneHotEncoding(feature:: AbstractArray{<:Any, 1}) 
    
    return oneHotEncoding(feature, unique(feature))
end

oneHotEncoding (generic function with 2 methods)

In [270]:
oneHot = oneHotEncoding(feature_2_cls)

13×1 Matrix{Bool}:
 1
 1
 0
 0
 1
 0
 1
 0
 1
 0
 1
 0
 1

In [ ]:
function oneHotEncoding(feature:: AbstractArray{Bool, 1}) 
    
    return reshape(feature, :, 1)
end

oneHotEncoding (generic function with 3 methods)

In [271]:
bool_feature = convert(Array{Bool, 1}, [1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1])
oneHot = oneHotEncoding(bool_feature)
typeof(oneHot)

Matrix{Bool} (alias for Array{Bool, 2})

## *Parte 2*: Normalización de datos

In [14]:
function calculateMinMaxNormalizationParameters(dataset:: AbstractArray{<:Real, 2})
    
    mins = minimum(dataset, dims=1)
    maxs = maximum(dataset, dims=1)
    return (mins, maxs)
end

calculateMinMaxNormalizationParameters (generic function with 1 method)

In [15]:
function calculateZeroMeanNormalizationParameters(dataset:: AbstractArray{<:Real, 2})
    
    meanValues = mean(dataset, dims=1)
    stdValues = std(dataset, dims=1)
    return (meanValues, stdValues)
end

calculateZeroMeanNormalizationParameters (generic function with 1 method)

In [155]:
function normalizeMinMax!(dataset:: AbstractArray{<: Real, 2}, normalizationParameters:: NTuple{2, AbstractArray{<: Real, 2}})
    
    mins, maxs = normalizationParameters
    dataset .-= mins
    dataset ./= (maxs .- mins)
    dataset[:, vec(mins.==maxs)] .= 0

end

normalizeMinMax! (generic function with 2 methods)

In [129]:
function normalizeMinMax!(dataset:: AbstractArray{<: Real, 2})
    
    normalizationParameters = calculateMinMaxNormalizationParameters(dataset)
    normalizeMinMax!(dataset, normalizationParameters)
end

normalizeMinMax! (generic function with 2 methods)

In [288]:
function normalizeMinMax(dataset:: AbstractArray{<: Real, 2}, normalizationParameters:: NTuple{2, AbstractArray{<: Real, 2}})
    
    mins, maxs = normalizationParameters
    normalized_dataset = copy(dataset)
    normalized_dataset .= (normalized_dataset .- mins) ./ (maxs .- mins)
    normalized_dataset[:, vec(mins.==maxs)] .= 0;
    return normalized_dataset

end

normalizeMinMax (generic function with 2 methods)

In [289]:
dataset = [5.1 4.5 6.1; 5.1 4.9 2.3; 5.1 4.1 2.0]
normvalues = calculateMinMaxNormalizationParameters(dataset)

normds = normalizeMinMax(dataset, normvalues)
println(dataset)
println(normds)


[5.1 4.5 6.1; 5.1 4.9 2.3; 5.1 4.1 2.0]
[0.0 0.5 1.0; 0.0 1.0 0.07317073170731704; 0.0 0.0 0.0]


In [290]:
function normalizeMinMax(dataset:: AbstractArray{<: Real, 2})
    
    normalizationParameters = calculateMinMaxNormalizationParameters(dataset)
    normalized_dataset = copy(dataset)
    return normalizeMinMax(normalized_dataset, normalizationParameters)
end

normalizeMinMax (generic function with 2 methods)

In [291]:
dataset = [5.1 4.5 6.1; 5.1 4.9 2.3; 5.1 4.1 2.0]

normds = normalizeMinMax(dataset)
println(dataset)
println(normds)

[5.1 4.5 6.1; 5.1 4.9 2.3; 5.1 4.1 2.0]
[0.0 0.5 1.0; 0.0 1.0 0.07317073170731704; 0.0 0.0 0.0]


In [178]:
function normalizeZeroMean!(dataset:: AbstractArray{<: Real, 2}, normalizationParameters:: NTuple{2, AbstractArray{<: Real, 2}})
    
    meanv, stdv = normvalues
    dataset .-= meanv
    dataset ./= stdv
    dataset[:, vec(stdv.==0)] .= 0;

end

normalizeZeroMean! (generic function with 2 methods)

In [292]:
dataset = [5.1 4.5 6.1; 5.1 4.9 2.3; 5.1 4.1 2.0]
normvalues = calculateZeroMeanNormalizationParameters(dataset)

normalizeZeroMean!(dataset, normvalues)
dataset

3×3 Matrix{Float64}:
 0.0   0.0   1.15221
 0.0   1.0  -0.510473
 0.0  -1.0  -0.641738

In [180]:
function normalizeZeroMean!(dataset:: AbstractArray{<: Real, 2})
    
    normvalues = calculateZeroMeanNormalizationParameters(dataset)
    normalizeZeroMean!(dataset, normvalues)
end

normalizeZeroMean! (generic function with 2 methods)

In [293]:
dataset = [5.1 4.5 6.1; 5.1 4.9 2.3; 5.1 4.1 2.0]
normvalues = calculateZeroMeanNormalizationParameters(dataset)

normalizeZeroMean!(dataset)
dataset

3×3 Matrix{Float64}:
 0.0   0.0   1.15221
 0.0   1.0  -0.510473
 0.0  -1.0  -0.641738

In [183]:
function normalizeZeroMean(dataset:: AbstractArray{<: Real, 2}, normalizationParameters:: NTuple{2, AbstractArray{<: Real, 2}})
    
    meanValues, stdValues = normalizationParameters
    normalized_dataset = copy(dataset)
    normalized_dataset .= (normalized_dataset .- meanValues) ./ stdValues
    normalized_dataset[:, vec(stdValues.==0)] .= 0;
    return normalized_dataset
end

normalizeZeroMean (generic function with 2 methods)

In [135]:
function normalizeZeroMean(dataset:: AbstractArray{<: Real, 2})
    
    norm_values = calculateZeroMeanNormalizationParameters(dataset)
    normalized_dataset = copy(dataset)
    return normalizeZeroMean(normalized_dataset, norm_values)
end

normalizeZeroMean (generic function with 2 methods)

## *Parte 3*: Clasificar outputs 

In [ ]:
function classifyOutputs(outputs::AbstractArray{<:Real,1}; threshold::Real=0.5)
    
    return outputs .>= threshold
end


classifyOutputs (generic function with 1 method)

In [190]:
outputs = [0.8, 0.2, 0.55, 0.6]
class = classifyOutputs(outputs, threshold = 0.2)

4-element BitVector:
 1
 1
 1
 1

In [204]:
function classifyOutputs(outputs::AbstractArray{<:Real,2}; threshold::Real=0.5) 

    dims = size(outputs)
    
    if dims[2] == 1
        return classifyOutputs(outputs[:]; threshold)
    else
        (_, indicesMaxEachInstance) = findmax(outputs, dims=2) # Obtener el índice de la clase con la probabilidad más alta
        outputs = falses(dims)
        outputs[indicesMaxEachInstance] .= true
        return outputs  
    end
end

classifyOutputs (generic function with 2 methods)

In [208]:
outputs = [0.1 0.8 0.5 0.6; 0.1 0.1 0.1 0.1; 0.8 0.1 0.4 0.3]
outputs'
classifyOutputs(outputs')

4×3 BitMatrix:
 0  0  1
 1  0  0
 1  0  0
 1  0  0

## *Parte 4*: Implementar métricas sobre los resultados de la ANN

In [ ]:
# Primer caso: se pasa un vector booleano para target y otro para outputs

function accuracy(outputs:: AbstractArray{Bool, 1}, targets:: AbstractArray{Bool, 1})
    
    return mean(targets .== outputs)
end


accuracy (generic function with 4 methods)

In [210]:
acc = accuracy(Bool.([1, 1, 0, 1, 0, 1]), Bool.([1, 0, 1, 0, 0, 1]))
acc

0.5

In [ ]:
# Segundo caso: se pasa una matriz de booleanos para targets y otra para outputs

function accuracy(outputs:: AbstractArray{Bool, 2}, targets:: AbstractArray{Bool, 2})

    dims = size(targets)

    if dims[2] == 1
        acc = accuracy(outputs[:], targets[:])
        println(acc)
        return acc

    elseif dims[2] > 2

        classComparison = targets .== outputs # Si la clase ha sido predicha correctamente, todos los elementos de la fila serán verdaderos (true == true, false == false, etc.)
        correctClassifications = all(classComparison, dims=2) 
        return mean(correctClassifications)

    end
end

accuracy (generic function with 4 methods)

In [243]:
outputs = Bool.([1 0 1 1 1 1 0; 0 0 0 0 0 1 1; 1 0 0 0 1 1 0])
targets = Bool.([1 0 1 0 1 1 0; 0 0 0 0 0 0 0; 0 1 0 1 0 0 1])


acc = accuracy(outputs', targets')

hola


0.14285714285714285

In [140]:
# Tercer caso: se pasa un vector de targets booleanos y un vector de probabilidades como salida de la ANN.

function accuracy(outputs::AbstractArray{<:Real,1}, targets::AbstractArray{Bool,1}; threshold::Real=0.5)
    
    return accuracy(outputs .>= threshold, targets)
end

accuracy (generic function with 3 methods)

In [247]:
outputs = [0.4, 0.9, 0.6, 0.3]
targets = Bool.([0, 1, 1, 0])

accuracy(outputs, targets; threshold=0.99)

hola qué tal


0.5

In [ ]:
# Cuarto caso: se pasa una matriz de targets booleanos y una matriz de probabilidades como salida de la ANN.

function accuracy(outputs::AbstractArray{<:Real,2}, targets::AbstractArray{Bool,2}; threshold::Real=0.5) 
           
    dims = size(targets)

    if dims[2] == 1
        return accuracy(outputs[:], targets[:]; threshold=threshold)

    elseif dims[2] > 2
        return accuracy(classifyOutputs(outputs; threshold=threshold), targets)
        
    end
end

accuracy (generic function with 4 methods)

In [265]:
outputs = Float32.([0.1 0.2 0.3 0.4 0.5 0.6 0.7; 0.0 0.0 0.0 0.0 0.1 0.1 0.1; 0.9 0.8 0.7 0.6 0.4 0.3 0.2])
targets = Bool.([1 0 1 0 1 1 0; 0 0 0 0 0 0 0; 0 1 0 1 0 0 1])


accuracy(outputs', targets')

im executing
hola


0.5714285714285714

## *Paso 5*: Crear una ANN

In [142]:
function buildClassANN(numInputs:: Int, topology:: AbstractArray{<:Int, 1}, numOutputs:: Int;
    transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)))
   
   ann = Chain()
   numInputsLayer = numInputs
   
   # Construir capas ocultas usando las funciones de transferencia correspondientes
   for (numOutputsLayer, transferFunction) in zip(topology, transferFunctions)
       ann = Chain(ann..., Dense(numInputsLayer, numOutputsLayer, transferFunction))
       numInputsLayer = numOutputsLayer
   end

   if numOutputs == 1 # problema de clasificacion binaria
       ann = Chain(ann..., Dense(numInputsLayer, numOutputs, σ));
   else # problema de clasificación multiclase
       ann = Chain(ann..., Dense(numInputsLayer, numOutputs), softmax);
   end

   return ann
end

buildClassANN (generic function with 1 method)

## *Paso 6*: Train ANN

In [143]:

function trainClassANN(topology:: AbstractArray{<: Int, 1}, dataset:: Tuple{AbstractArray{<: Real, 2}, AbstractArray{Bool, 2}};
    transferFunctions::AbstractArray{<:Function,1} = fill(σ, length(topology)), maxEpochs:: Int = 1000, minLoss:: Real = 0.0, learningRate:: Real = 0.01)

   inputs, targets = dataset
   inputs = Float32.(inputs')  # Convertir a Float32 y trasponer
   targets = (targets')   # Trasponer
   
   numInputs = size(inputs, 1)
   numOutputs = size(targets, 1)
   
   ann = buildClassANN(numInputs, topology, numOutputs; transferFunctions) # build ANN
   loss(model, x,y) = (size(y,1) == 1) ? Losses.binarycrossentropy(model(x),y) : Losses.crossentropy(model(x),y) # loss function
   opt_state = Flux.setup(ADAM(learningRate), ann) # optimizer

   losses = Float32[] # losses array
   push!(losses, loss(ann, inputs, targets)) # append iteration 0 loss 
   
   for epoch in 1:maxEpochs
       
       Flux.train!(loss, ann, [(inputs, targets)], opt_state)
       current_loss = loss(ann, inputs, targets)
       push!(losses, current_loss)
       
       if current_loss ≤ minLoss
           break
       end
   end
   
   return ann, losses
end

trainClassANN (generic function with 1 method)

In [144]:
function trainClassANN(topology:: AbstractArray{<: Int, 1}, (inputs, targets):: Tuple{AbstractArray{<: Real, 2}, AbstractArray{Bool, 1}};
    transferFunctions::AbstractArray{<:Function,1} = fill(σ, length(topology)), maxEpochs:: Int = 1000, minLoss:: Real = 0.0, learningRate:: Real = 0.01)

   reshape!(targets, :, 1) 
   return trainClassANN(topology, (inputs, targets); transferFunctions=transferFunctions, maxEpochs=maxEpochs, minLoss=minLoss, learningRate=learningRate)
   
end

trainClassANN (generic function with 2 methods)

## *Final Step*: Basic Pipeline

In [145]:
dataset = readdlm("iris.data", ',')

# Asegurarnos de que los inputs son AbstractArray{<:Real, 2}
inputs = convert(AbstractArray{Float32, 2}, normalizeZeroMean(Float32.(dataset[:, 1:4])))

# Asegurarnos de que outputs es AbstractArray{Bool, 2}
outputs = convert(AbstractArray{Bool, 2}, oneHotEncoding(dataset[:, 5]))

# Asegurarnos de que topology es AbstractArray{<:Int, 1}
topology = convert(AbstractArray{Int, 1}, [8, 4])

# Para debuggear, vamos a imprimir los tipos
println("Type of inputs: ", typeof(inputs))
println("Type of outputs: ", typeof(outputs))
println("Type of topology: ", typeof(topology))

# Llamada a la función
ann, loss = trainClassANN(topology, (inputs, outputs), [σ, σ], 1000, 0.001, 0.1)


ArgumentError: ArgumentError: Cannot open 'iris.data': not a file

In [146]:
using Plots

function plotLoss(losses)
    plot(losses, 
         label="Training Loss", 
         xlabel="Epoch", 
         ylabel="Loss",
         title="Loss Function Evolution",
         linewidth=2,
         legend=:topright)
end

# Usar la función
plotLoss(loss)

#0.0009752035

UndefVarError: UndefVarError: `loss` not defined in `Main`
Suggestion: check for spelling errors or missing imports.